# Reto Semana 5: Perfilador de Datasets

## Programacion para Ciencia de Datos
**Instituto Politecnico Nacional** | Semestre Febrero-Julio 2026

---

## El Escenario del Mundo Real

Recibes datasets de diferentes fuentes: marketing, ventas, recursos humanos, sensores IoT... Cada uno tiene diferentes columnas, tipos de datos y problemas de calidad.

Antes de analizar cualquier dataset, necesitas responder:
- Que columnas tiene?
- Que tipo de datos contiene cada columna?
- Cuantos valores faltan?
- Hay datos duplicados?

Tu mision: crear una **herramienta reutilizable** que perfila **cualquier CSV** automaticamente.

```
╔════════════════════════════════════════════════════════════════════════════╗
║                      PERFILADOR DE DATASETS                                ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║   CUALQUIER CSV                      REPORTE DE CALIDAD                    ║
║   ─────────────                      ──────────────────                    ║
║                                                                            ║
║   ventas.csv          ┌───────────┐   nombre_columna,tipo,nulos,...        ║
║   empleados.csv   ──> │ PERFILADOR│ ─> fecha,fecha,0,0.00%,...             ║
║   sensores.csv        │ UNIVERSAL │   producto,texto,5,2.50%,...           ║
║   cualquier.csv       └───────────┘   cantidad,numerico,10,5.00%,...       ║
║                                                                            ║
║   python main.py --input data/X.csv --output outputs/perfil_X.csv          ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
```

---

## Especificacion de Entrada

### Argumentos de Linea de Comandos

```bash
python main.py --input <ruta_csv_entrada> --output <ruta_csv_salida>
```

| Argumento | Requerido | Descripcion |
|-----------|-----------|-------------|
| `--input` o `-i` | Si | Ruta al archivo CSV a perfilar |
| `--output` o `-o` | Si | Ruta donde guardar el perfil |

### Formato del CSV de Entrada

- Cualquier archivo CSV valido
- Primera fila: encabezados (nombres de columnas)
- Separador: coma (`,`)
- Codificacion: UTF-8

### Ejemplos de Entrada

El perfilador debe funcionar con **cualquier** CSV:

In [ ]:
# Ejemplo 1: Datos de ventas
ventas_csv = '''fecha,producto,cantidad,precio,vendedor
2026-01-01,Laptop,2,15000.00,Ana
2026-01-02,Mouse,10,250.00,Bob
2026-01-03,Teclado,,800.00,Ana
2026-01-04,Monitor,3,,Carlos
2026-01-05,Laptop,1,15000.00,
'''

print("=== ventas.csv ===")
print(ventas_csv)

In [ ]:
# Ejemplo 2: Datos de empleados
empleados_csv = '''id,nombre,email,departamento,salario,activo
1,Ana Garcia,ana@empresa.com,TI,45000.00,true
2,Bob Lopez,,Ventas,38000.00,true
3,Carlos Ruiz,carlos@empresa.com,TI,52000.00,false
4,,diana@empresa.com,RRHH,41000.00,true
5,Eva Torres,eva@empresa.com,,47000.00,true
'''

print("=== empleados.csv ===")
print(empleados_csv)

In [ ]:
# Ejemplo 3: Datos de sensores IoT
sensores_csv = '''timestamp,sensor_id,temperatura,humedad,bateria
2026-01-01 10:00:00,S001,23.5,65.2,98
2026-01-01 10:05:00,S001,23.7,64.8,98
2026-01-01 10:10:00,S002,22.1,,97
2026-01-01 10:15:00,S001,,65.5,97
2026-01-01 10:20:00,S003,25.2,70.1,
'''

print("=== sensores.csv ===")
print(sensores_csv)

---

## Especificacion de Salida

### Columnas del Perfil

Para **cada columna** del CSV de entrada, calcular:

| Columna de Salida | Tipo | Descripcion |
|-------------------|------|-------------|
| `nombre_columna` | texto | Nombre de la columna analizada |
| `tipo_inferido` | texto | "numerico", "texto", "fecha", "booleano" |
| `total_registros` | entero | Cantidad total de filas (sin contar encabezado) |
| `valores_nulos` | entero | Cantidad de valores vacios o nulos |
| `porcentaje_nulos` | decimal | (nulos / total) * 100, con 2 decimales |
| `valores_unicos` | entero | Cantidad de valores distintos (sin contar nulos) |
| `porcentaje_unicos` | decimal | (unicos / total) * 100, con 2 decimales |
| `ejemplo_valor` | texto | Primer valor no nulo encontrado |

### Formato de Salida

- CSV con una fila por columna del archivo original
- Primera fila: encabezados
- Orden: mismo orden que las columnas en el archivo original

---

## Reglas de Procesamiento

### Regla 1: Deteccion de Valores Nulos

Se considera **nulo/vacio**:
- Celda completamente vacia: `,,`
- Celda con solo espacios: `, ,`
- String vacio: `""`

**NO** son nulos:
- El numero `0`
- El string `"0"`
- El string `"null"` o `"None"` (son texto literal)

In [ ]:
def es_valor_nulo(valor):
    """
    Determina si un valor se considera nulo.
    
    Nulo: None, string vacio, string con solo espacios
    NO nulo: 0, "0", "null", "None", cualquier otro texto
    """
    if valor is None:
        return True
    if isinstance(valor, str) and valor.strip() == "":
        return True
    return False

# Pruebas
casos = [
    ("", True),
    ("  ", True),
    (None, True),
    ("0", False),
    (0, False),
    ("null", False),
    ("None", False),
    ("Ana", False),
    ("123", False),
]

print("Pruebas de deteccion de nulos:")
for valor, esperado in casos:
    resultado = es_valor_nulo(valor)
    estado = "OK" if resultado == esperado else "ERROR"
    print(f"  es_valor_nulo({repr(valor):>10}) = {resultado!s:>5} [{estado}]")

### Regla 2: Inferencia de Tipo de Dato

Analiza los valores **no nulos** de la columna y determina el tipo predominante:

| Tipo | Condicion |
|------|------|
| `numerico` | >80% de valores se pueden convertir a float |
| `fecha` | >80% de valores tienen formato de fecha (YYYY-MM-DD) |
| `booleano` | >80% de valores son true/false/yes/no/si/1/0 |
| `texto` | Cualquier otro caso |

In [ ]:
def es_numerico(valor):
    """Verifica si un valor es numerico."""
    try:
        float(str(valor).replace(',', '').strip())
        return True
    except (ValueError, TypeError):
        return False


def es_fecha(valor):
    """Verifica si un valor parece una fecha YYYY-MM-DD."""
    v = str(valor).strip()
    
    # Formato YYYY-MM-DD
    if len(v) >= 10:
        fecha_parte = v[:10]
        if fecha_parte[4] == '-' and fecha_parte[7] == '-':
            try:
                partes = fecha_parte.split('-')
                año = int(partes[0])
                mes = int(partes[1])
                dia = int(partes[2])
                if 1900 <= año <= 2100 and 1 <= mes <= 12 and 1 <= dia <= 31:
                    return True
            except (ValueError, IndexError):
                pass
    return False


def es_booleano(valor):
    """Verifica si un valor es booleano."""
    v = str(valor).strip().lower()
    return v in ['true', 'false', 'yes', 'no', 'si', 'no', '1', '0', 't', 'f']


def inferir_tipo_columna(valores):
    """
    Infiere el tipo de una columna basado en sus valores.
    
    Returns:
        str: 'numerico', 'fecha', 'booleano', o 'texto'
    """
    # Filtrar valores no nulos
    valores_validos = [v for v in valores if not es_valor_nulo(v)]
    
    if not valores_validos:
        return "texto"  # Si todo es nulo, asumimos texto
    
    total = len(valores_validos)
    umbral = 0.8  # 80% para determinar el tipo
    
    # Contar cada tipo
    num_numericos = sum(1 for v in valores_validos if es_numerico(v))
    num_fechas = sum(1 for v in valores_validos if es_fecha(v))
    num_booleanos = sum(1 for v in valores_validos if es_booleano(v))
    
    # Determinar tipo predominante
    if num_fechas / total >= umbral:
        return "fecha"
    elif num_booleanos / total >= umbral:
        return "booleano"
    elif num_numericos / total >= umbral:
        return "numerico"
    else:
        return "texto"


# Pruebas
print("Pruebas de inferencia de tipo:")
print(f"  [1, 2, 3, 4, 5]: {inferir_tipo_columna(['1', '2', '3', '4', '5'])}")
print(f"  [Ana, Bob, Carlos]: {inferir_tipo_columna(['Ana', 'Bob', 'Carlos'])}")
print(f"  [2026-01-01, 2026-01-02]: {inferir_tipo_columna(['2026-01-01', '2026-01-02', '2026-01-03'])}")
print(f"  [true, false, true]: {inferir_tipo_columna(['true', 'false', 'true', 'yes', 'no'])}")
print(f"  [mezcla]: {inferir_tipo_columna(['Ana', '123', '2026-01-01'])}")

### Regla 3: Valores Unicos

- Contar solo valores **no nulos**
- Cada valor distinto cuenta una vez
- Case-sensitive: "Ana" y "ana" son diferentes

In [ ]:
def contar_unicos(valores):
    """Cuenta valores unicos, excluyendo nulos."""
    valores_no_nulos = [v for v in valores if not es_valor_nulo(v)]
    return len(set(valores_no_nulos))

# Prueba
ejemplo = ["Ana", "Bob", "Ana", "", "Carlos", "Bob", None, "Diana"]
print(f"Valores: {ejemplo}")
print(f"Unicos (sin nulos): {contar_unicos(ejemplo)}")
# Deberia ser 4: Ana, Bob, Carlos, Diana

### Regla 4: Formato de Porcentajes

- Siempre **2 decimales**
- Sin simbolo de porcentaje en el CSV
- Ejemplos: `0.00`, `25.50`, `100.00`

In [ ]:
def calcular_porcentaje(parte, total):
    """Calcula porcentaje con 2 decimales."""
    if total == 0:
        return 0.00
    return round((parte / total) * 100, 2)

# Pruebas
print(f"5 de 20: {calcular_porcentaje(5, 20)}%")    # 25.00
print(f"1 de 3: {calcular_porcentaje(1, 3)}%")      # 33.33
print(f"0 de 10: {calcular_porcentaje(0, 10)}%")    # 0.00
print(f"0 de 0: {calcular_porcentaje(0, 0)}%")      # 0.00

---

## Ejemplo Completo de Entrada/Salida

### Entrada: `data/ventas.csv`

```csv
fecha,producto,cantidad,precio,vendedor
2026-01-01,Laptop,2,15000.00,Ana
2026-01-02,Mouse,10,250.00,Bob
2026-01-03,Teclado,,800.00,Ana
2026-01-04,Monitor,3,,Carlos
2026-01-05,Laptop,1,15000.00,
```

### Ejecucion

```bash
python main.py --input data/ventas.csv --output outputs/perfil_ventas.csv
```

### Salida: `outputs/perfil_ventas.csv`

```csv
nombre_columna,tipo_inferido,total_registros,valores_nulos,porcentaje_nulos,valores_unicos,porcentaje_unicos,ejemplo_valor
fecha,fecha,5,0,0.00,5,100.00,2026-01-01
producto,texto,5,0,0.00,4,80.00,Laptop
cantidad,numerico,5,1,20.00,4,80.00,2
precio,numerico,5,1,20.00,3,60.00,15000.00
vendedor,texto,5,1,20.00,3,60.00,Ana
```

In [ ]:
# Implementacion completa del perfilador

def perfilar_columna(nombre, valores):
    """
    Genera el perfil completo de una columna.
    
    Args:
        nombre: Nombre de la columna
        valores: Lista de valores de la columna
    
    Returns:
        dict: Perfil de la columna
    """
    total = len(valores)
    nulos = sum(1 for v in valores if es_valor_nulo(v))
    valores_no_nulos = [v for v in valores if not es_valor_nulo(v)]
    unicos = len(set(valores_no_nulos))
    ejemplo = valores_no_nulos[0] if valores_no_nulos else ""
    tipo = inferir_tipo_columna(valores)
    
    return {
        "nombre_columna": nombre,
        "tipo_inferido": tipo,
        "total_registros": total,
        "valores_nulos": nulos,
        "porcentaje_nulos": calcular_porcentaje(nulos, total),
        "valores_unicos": unicos,
        "porcentaje_unicos": calcular_porcentaje(unicos, total),
        "ejemplo_valor": ejemplo
    }


def perfilar_csv(contenido_csv):
    """
    Perfila un CSV completo.
    
    Args:
        contenido_csv: String con el contenido del CSV
    
    Returns:
        list: Lista de perfiles (uno por columna)
    """
    lineas = contenido_csv.strip().split('\n')
    
    if len(lineas) < 2:
        return []
    
    # Parsear encabezados y datos
    encabezados = lineas[0].split(',')
    filas = [linea.split(',') for linea in lineas[1:]]
    
    perfiles = []
    
    for i, nombre_col in enumerate(encabezados):
        # Extraer valores de esta columna
        valores = []
        for fila in filas:
            if i < len(fila):
                valores.append(fila[i])
            else:
                valores.append("")  # Columna faltante
        
        perfil = perfilar_columna(nombre_col, valores)
        perfiles.append(perfil)
    
    return perfiles


# Probar con datos de ventas
ventas_csv = '''fecha,producto,cantidad,precio,vendedor
2026-01-01,Laptop,2,15000.00,Ana
2026-01-02,Mouse,10,250.00,Bob
2026-01-03,Teclado,,800.00,Ana
2026-01-04,Monitor,3,,Carlos
2026-01-05,Laptop,1,15000.00,'''

perfiles = perfilar_csv(ventas_csv)

print("=" * 70)
print("PERFIL DE ventas.csv")
print("=" * 70)

for p in perfiles:
    print(f"\n{p['nombre_columna']}:")
    print(f"  Tipo: {p['tipo_inferido']}")
    print(f"  Total: {p['total_registros']} | Nulos: {p['valores_nulos']} ({p['porcentaje_nulos']}%)")
    print(f"  Unicos: {p['valores_unicos']} ({p['porcentaje_unicos']}%)")
    print(f"  Ejemplo: '{p['ejemplo_valor']}'")

In [ ]:
# Generar CSV de salida

def generar_csv_salida(perfiles):
    """Genera el string CSV de salida."""
    columnas = [
        "nombre_columna", "tipo_inferido", "total_registros",
        "valores_nulos", "porcentaje_nulos", "valores_unicos",
        "porcentaje_unicos", "ejemplo_valor"
    ]
    
    lineas = [",".join(columnas)]  # Encabezados
    
    for p in perfiles:
        valores = [
            str(p["nombre_columna"]),
            str(p["tipo_inferido"]),
            str(p["total_registros"]),
            str(p["valores_nulos"]),
            f"{p['porcentaje_nulos']:.2f}",
            str(p["valores_unicos"]),
            f"{p['porcentaje_unicos']:.2f}",
            str(p["ejemplo_valor"])
        ]
        lineas.append(",".join(valores))
    
    return "\n".join(lineas)


# Generar salida
csv_salida = generar_csv_salida(perfiles)

print("\n" + "=" * 70)
print("CSV DE SALIDA:")
print("=" * 70)
print(csv_salida)

---

## Estructura del Proyecto

```
reto_semana_05/
│
├── main.py                 # Programa principal
├── requirements.txt        # Dependencias (puede estar vacio)
├── README.md               # Instrucciones de instalacion y uso
├── .gitignore              # Archivos a ignorar
│
├── data/                   # CSVs de prueba
│   ├── ventas.csv
│   ├── empleados.csv
│   └── sensores.csv
│
└── outputs/                # Perfiles generados
    └── (se generan al ejecutar)
```

---

## Codigo de main.py Completo

In [ ]:
main_py = '''
#!/usr/bin/env python3
"""
Perfilador de Datasets CSV
Analiza cualquier archivo CSV y genera un reporte de calidad de datos.

Uso:
    python main.py --input <archivo.csv> --output <perfil.csv>
"""

import argparse
import sys


def es_valor_nulo(valor):
    """Determina si un valor se considera nulo."""
    if valor is None:
        return True
    if isinstance(valor, str) and valor.strip() == "":
        return True
    return False


def es_numerico(valor):
    """Verifica si un valor es numerico."""
    try:
        float(str(valor).replace(\',\', \'\').strip())
        return True
    except (ValueError, TypeError):
        return False


def es_fecha(valor):
    """Verifica si un valor parece una fecha YYYY-MM-DD."""
    v = str(valor).strip()
    if len(v) >= 10 and v[4] == \'-\' and v[7] == \'-\':
        try:
            partes = v[:10].split(\'-\')
            año, mes, dia = int(partes[0]), int(partes[1]), int(partes[2])
            return 1900 <= año <= 2100 and 1 <= mes <= 12 and 1 <= dia <= 31
        except (ValueError, IndexError):
            pass
    return False


def es_booleano(valor):
    """Verifica si un valor es booleano."""
    v = str(valor).strip().lower()
    return v in [\'true\', \'false\', \'yes\', \'no\', \'si\', \'1\', \'0\', \'t\', \'f\']


def inferir_tipo(valores):
    """Infiere el tipo de una columna."""
    valores_validos = [v for v in valores if not es_valor_nulo(v)]
    if not valores_validos:
        return "texto"
    
    total = len(valores_validos)
    umbral = 0.8
    
    num_fechas = sum(1 for v in valores_validos if es_fecha(v))
    num_booleanos = sum(1 for v in valores_validos if es_booleano(v))
    num_numericos = sum(1 for v in valores_validos if es_numerico(v))
    
    if num_fechas / total >= umbral:
        return "fecha"
    elif num_booleanos / total >= umbral:
        return "booleano"
    elif num_numericos / total >= umbral:
        return "numerico"
    else:
        return "texto"


def perfilar_columna(nombre, valores):
    """Genera el perfil de una columna."""
    total = len(valores)
    nulos = sum(1 for v in valores if es_valor_nulo(v))
    valores_no_nulos = [v for v in valores if not es_valor_nulo(v)]
    unicos = len(set(valores_no_nulos))
    ejemplo = valores_no_nulos[0] if valores_no_nulos else ""
    tipo = inferir_tipo(valores)
    
    pct_nulos = round(nulos / total * 100, 2) if total > 0 else 0.00
    pct_unicos = round(unicos / total * 100, 2) if total > 0 else 0.00
    
    return {
        "nombre_columna": nombre,
        "tipo_inferido": tipo,
        "total_registros": total,
        "valores_nulos": nulos,
        "porcentaje_nulos": pct_nulos,
        "valores_unicos": unicos,
        "porcentaje_unicos": pct_unicos,
        "ejemplo_valor": ejemplo
    }


def leer_csv(ruta):
    """Lee un archivo CSV y retorna encabezados y filas."""
    with open(ruta, \'r\', encoding=\'utf-8\') as f:
        lineas = f.readlines()
    
    if not lineas:
        return [], []
    
    encabezados = lineas[0].strip().split(\',\')
    filas = [linea.strip().split(\',\') for linea in lineas[1:] if linea.strip()]
    
    return encabezados, filas


def escribir_csv(ruta, perfiles):
    """Escribe el CSV de perfiles."""
    columnas = [
        "nombre_columna", "tipo_inferido", "total_registros",
        "valores_nulos", "porcentaje_nulos", "valores_unicos",
        "porcentaje_unicos", "ejemplo_valor"
    ]
    
    with open(ruta, \'w\', encoding=\'utf-8\') as f:
        f.write(\',\'.join(columnas) + \'\\n\')
        
        for p in perfiles:
            valores = [
                str(p["nombre_columna"]),
                str(p["tipo_inferido"]),
                str(p["total_registros"]),
                str(p["valores_nulos"]),
                f"{p[\'porcentaje_nulos\']:.2f}",
                str(p["valores_unicos"]),
                f"{p[\'porcentaje_unicos\']:.2f}",
                str(p["ejemplo_valor"])
            ]
            f.write(\',\'.join(valores) + \'\\n\')


def main():
    # Parsear argumentos
    parser = argparse.ArgumentParser(
        description="Perfilador de Datasets CSV"
    )
    parser.add_argument("--input", "-i", required=True, 
                        help="Ruta al CSV de entrada")
    parser.add_argument("--output", "-o", required=True,
                        help="Ruta al CSV de salida")
    
    args = parser.parse_args()
    
    print(f"Perfilando: {args.input}")
    
    # Leer CSV
    try:
        encabezados, filas = leer_csv(args.input)
    except FileNotFoundError:
        print(f"Error: No se encontro el archivo {args.input}")
        sys.exit(1)
    
    if not encabezados:
        print("Error: El archivo esta vacio")
        sys.exit(1)
    
    print(f"Columnas encontradas: {len(encabezados)}")
    print(f"Registros: {len(filas)}")
    
    # Perfilar cada columna
    perfiles = []
    for i, nombre_col in enumerate(encabezados):
        valores = [fila[i] if i < len(fila) else "" for fila in filas]
        perfil = perfilar_columna(nombre_col, valores)
        perfiles.append(perfil)
    
    # Escribir resultado
    escribir_csv(args.output, perfiles)
    print(f"Perfil guardado en: {args.output}")
    print("Completado!")


if __name__ == "__main__":
    main()
'''

print(main_py)

---

## Requisitos de Reproducibilidad

### requirements.txt

```
# requirements.txt
# Este proyecto no tiene dependencias externas
# (usa solo la biblioteca estandar de Python)
```

**Nota**: Si no hay dependencias externas, el archivo puede estar vacio o tener solo comentarios. Pero **debe existir**.

### README.md Requerido

In [ ]:
readme_requerido = '''
# Perfilador de Datasets

Herramienta que analiza archivos CSV y genera reportes de calidad de datos.

## Requisitos

- Python 3.8 o superior

## Instalacion

### 1. Clonar el repositorio
```bash
git clone https://github.com/usuario/reto-semana-05.git
cd reto-semana-05
```

### 2. Crear ambiente virtual
```bash
python -m venv .venv
```

### 3. Activar ambiente virtual
```bash
# Windows
.venv\\Scripts\\activate

# Linux/Mac
source .venv/bin/activate
```

### 4. Instalar dependencias
```bash
pip install -r requirements.txt
```

## Uso

```bash
python main.py --input <archivo_entrada.csv> --output <archivo_salida.csv>
```

### Ejemplo
```bash
python main.py --input data/ventas.csv --output outputs/perfil_ventas.csv
```

## Formato de Salida

El perfil generado contiene las siguientes columnas:

| Columna | Descripcion |
|---------|-------------|
| nombre_columna | Nombre de la columna |
| tipo_inferido | Tipo detectado (numerico/texto/fecha/booleano) |
| total_registros | Total de filas |
| valores_nulos | Cantidad de valores vacios |
| porcentaje_nulos | Porcentaje de nulos |
| valores_unicos | Cantidad de valores distintos |
| porcentaje_unicos | Porcentaje de unicidad |
| ejemplo_valor | Primer valor no nulo |

## Autor

[Tu nombre] - Febrero 2026
'''

print(readme_requerido)

---

## Criterios de Evaluacion

| Criterio | Puntos |
|----------|--------|
| Argumentos de linea de comandos funcionan | 10 |
| Lectura correcta de cualquier CSV | 15 |
| Deteccion correcta de nulos | 15 |
| Inferencia de tipo correcta | 15 |
| Calculo de metricas correcto | 15 |
| Formato de salida exacto | 10 |
| **Reproducibilidad** | 15 |
| - requirements.txt existe | 5 |
| - README con instrucciones completas | 5 |
| - Proyecto funciona siguiendo README | 5 |
| Git (commits significativos) | 5 |
| **Total** | **100** |

### Test de Reproducibilidad

El evaluador hara exactamente esto:

```bash
git clone <tu-repo>
cd <tu-repo>
python -m venv .venv
.venv\Scripts\activate  # o source .venv/bin/activate
pip install -r requirements.txt
python main.py --input data/test.csv --output outputs/test_perfil.csv
```

**Si esto no funciona, pierdes los 15 puntos de reproducibilidad.**

---

## Fecha de Entrega

**Viernes de la Semana 5, 23:59 hrs**

---

*Reto Semana 5 - Programacion para Ciencia de Datos - IPN 2026*